# DSA5206 Part II expanded analysis

94 images, 65 patients, four frozen encoders and 36 actual model configurations. The analysis below reads and checks the completed outputs. The 2020 study is assessed using the 12 questions in Lecture 9. See the report and checklist.csv for evidence.

The extension is exploratory and follows inspection of the initial experiment. Individual best test results are labelled as observed rankings. Model-family selection in the three selectors uses training folds only.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
ROOT=Path.cwd()
assert (ROOT/'data/scored_metadata.csv').exists()
sys.path.insert(0,str(ROOT/'code'))
meta=pd.read_csv(ROOT/'data/scored_metadata.csv')
print(len(meta),'images;',meta.patientid.nunique(),'patients')
print(json.loads((ROOT/'results/expanded/verification.json').read_text()))

94 images; 65 patients
{'n_images': 94, 'n_patients': 65, 'initial_outer_fits': 900, 'expanded_half_outer_fits': 2700, 'expanded_fivefold_outer_fits': 270, 'selector_half_evaluations': 300, 'selector_fivefold_evaluations': 30, 'total_actual_outer_fits': 3870, 'all_half_metric_rows': 3900, 'best': {'Geographic Extent': {'original_reference': 'Published OLS 1', 'bounded_reference': 'Bounded OLS 1', 'best_modern': 'BioViL PCA ridge', 'best_alternative': 'BioViL PCA ridge', 'lowest_observed_model': 'BioViL PCA ridge'}, 'Opacity': {'original_reference': 'Published OLS 4', 'bounded_reference': 'Bounded OLS 4', 'best_modern': 'BioViL-T PCA ridge', 'best_alternative': 'BioViL-T PCA ridge', 'lowest_observed_model': 'BioViL-T PCA ridge'}}, 'protocol_sha256': 'f8a38a7780c0b45b209281ba06515023b79e0e53c9e2bb49c950c72dabe6e02c', 'fivefold_predictions_unique': True, 'all_predictions_finite': True, 'model_selection_uses': 'inner training CV only; individual leaderboard rankings remain exploratory'}


## Check grouping and recompute stored metrics

These checks verify outer and inner patient separation, feature hashes, target alignment, all recorded errors, model-family selection and the low-dimensional OLS clipping ablation and the high-dimensional thread sensitivity.

In [2]:
from verify_results import main as verify_initial
from verify_extended import main as verify_extension
verify_initial()
verify_extension()

PASS: feature checksum; 94 images and 65 patients; no train/test patient overlap; all 900 recorded MAE, MSE and R2 values match predictions.
PASS: 3300 held-out metric rows recomputed; patient separation, target alignment, bounds, training-only selection and feature hashes verified. OLS 1/4/18 raw fits match the initial run; OLS 1024 differs by at most 0.001 under the audited BLAS thread change.


## Complete leaderboard

All configurations are shown, including unsuccessful alternatives. MAE is mean image absolute error. The SD describes dependent resampling partitions and is not a confidence interval.

In [3]:
summary=pd.read_csv(ROOT/'results/expanded/all_models_summary.csv')
print(summary.sort_values(['target','MAE_mean'])[['target','model','MAE_mean','MAE_std','Patient_MAE_mean','R2_mean']].round(4).to_string(index=False))

           target                       model  MAE_mean  MAE_std  Patient_MAE_mean  R2_mean
Geographic Extent            BioViL PCA ridge    0.7627   0.0735            0.7869   0.8198
Geographic Extent          BioViL-T PCA ridge    0.7912   0.0947            0.8059   0.8121
Geographic Extent       BioViL-T pooled ridge    0.8012   0.1103            0.8092   0.8074
Geographic Extent         BioViL pooled ridge    0.8245   0.0930            0.8504   0.7960
Geographic Extent    Training selected modern    0.8274   0.1200            0.8472   0.7905
Geographic Extent       Training selected all    0.8274   0.1200            0.8472   0.7905
Geographic Extent      BioViL-T projected SVR    0.8605   0.0890            0.8794   0.7832
Geographic Extent        BioViL projected SVR    0.9268   0.0880            0.9720   0.7260
Geographic Extent    BioViL-T projected ridge    0.9295   0.1129            0.9562   0.7486
Geographic Extent      BioViL projected ridge    0.9673   0.1072            1.00

## Training-only choice and paired comparisons

A candidate winner identified by the outer-test leaderboard is exploratory. The selectors instead choose their family from inner CV before examining that split's test predictions. Negative MAE differences favor the named model. No independent-trials significance test is applied to the overlapping splits.

In [4]:
paired=pd.read_csv(ROOT/'results/expanded/paired_comparisons.csv')
sel=paired[paired.model.str.startswith('Training selected') & ((paired.target.eq('Geographic Extent') & paired.reference.eq('Bounded OLS 1')) | (paired.target.eq('Opacity') & paired.reference.eq('Bounded OLS 4')))]
print(sel[['target','reference','model','MAE_difference','split_win_fraction','patient_error_difference','conditional_bootstrap_low','conditional_bootstrap_high']].round(4).to_string(index=False))

           target     reference                    model  MAE_difference  split_win_fraction  patient_error_difference  conditional_bootstrap_low  conditional_bootstrap_high
Geographic Extent Bounded OLS 1 Training selected modern         -0.2796                0.96                   -0.2142                    -0.4219                     -0.0188
Geographic Extent Bounded OLS 1    Training selected all         -0.2796                0.96                   -0.2142                    -0.4172                     -0.0188
Geographic Extent Bounded OLS 1    Training selected OLS          0.0197                0.10                    0.0204                    -0.0109                      0.0540
          Opacity Bounded OLS 4 Training selected modern         -0.0445                0.68                   -0.0516                    -0.1693                      0.0632
          Opacity Bounded OLS 4    Training selected all         -0.0021                0.50                   -0.0105            

## Larger training sets in fivefold CV

Each patient is held out once. The pooled error uses 94 unique image predictions per configuration. This is an internal sensitivity check on the same patients, not a new external cohort.

In [5]:
five=pd.read_csv(ROOT/'results/expanded/fivefold_pooled_summary.csv')
print(five[five.model.str.startswith('Training selected')].round(4).to_string(index=False))

           target                    model    MAE    MSE     R2  Patient_MAE    Bias
Geographic Extent    Training selected OLS 1.1176 2.0348 0.6338       1.1073  0.0657
Geographic Extent Training selected modern 0.7537 0.9652 0.8263       0.7768 -0.0742
Geographic Extent    Training selected all 0.7537 0.9652 0.8263       0.7768 -0.0742
          Opacity    Training selected OLS 0.7227 0.8259 0.6365       0.7530  0.0103
          Opacity Training selected modern 0.6575 0.6589 0.7100       0.6784 -0.0191
          Opacity    Training selected all 0.6575 0.6589 0.7100       0.6784 -0.0191


## Score bounds and the diagnosed scaling problem

All new candidates and new baseline references use the known score range in inner scoring and testing. Initial raw outcomes remain available. Comparing raw and bounded OLS isolates the simple clipping effect. The unscaled, filtered and PCA Ridge variants change more than one part of the original failed pipeline, so their gain is a pipeline gain.

In [6]:
print(pd.read_csv(ROOT/'results/expanded/ablation_summary.csv')[['target','model','MAE_mean','MAE_std']].round(4).to_string(index=False))
print(json.loads((ROOT/'results/failure_diagnostic.json').read_text()))

           target                       model  MAE_mean  MAE_std
Geographic Extent             Published OLS 1    1.1652   0.1040
Geographic Extent             Published OLS 4    1.1837   0.1206
Geographic Extent                  Ridge 1024    1.4186   0.6332
Geographic Extent               Bounded OLS 1    1.1070   0.1103
Geographic Extent               Bounded OLS 4    1.1168   0.1142
Geographic Extent DenseNet1024 unscaled ridge    1.1523   0.1382
Geographic Extent DenseNet1024 filtered ridge    1.1463   0.1355
Geographic Extent      DenseNet1024 PCA ridge    1.1772   0.1525
          Opacity             Published OLS 1    0.7884   0.0621
          Opacity             Published OLS 4    0.7663   0.0722
          Opacity                  Ridge 1024    0.9310   0.5151
          Opacity               Bounded OLS 1    0.7721   0.0667
          Opacity               Bounded OLS 4    0.7453   0.0720
          Opacity DenseNet1024 unscaled ridge    0.8132   0.0960
          Opacity DenseNe

## High-score errors

Extent >=6 and opacity >=4 are exploratory imaging strata. They are not validated clinical severity thresholds.

In [7]:
names=['Training selected OLS','Training selected modern','Training selected all']
print(summary[summary.model.isin(names)][['target','model','MAE_mean','High_score_MAE_mean','High_score_bias_mean']].round(4).to_string(index=False))

           target                    model  MAE_mean  High_score_MAE_mean  High_score_bias_mean
Geographic Extent    Training selected OLS    1.1267               1.5370               -1.4475
Geographic Extent Training selected modern    0.8274               0.6931               -0.4216
Geographic Extent    Training selected all    0.8274               0.6931               -0.4216
          Opacity    Training selected OLS    0.7659               0.8448               -0.7447
          Opacity Training selected modern    0.7009               0.7906               -0.3691
          Opacity    Training selected all    0.7432               0.8579               -0.4656


## Optional full regression rerun

Set RERUN=True to rerun from cached features. Individual checkpoint files, when present, are reused. The released package does not ship those redundant local checkpoints. Re-extracting the encoders requires requirements-images.txt and code/prepare_extended_features.py.

In [8]:
RERUN=False
if RERUN:
    import subprocess
    subprocess.run([sys.executable,'code/run_extended_comparison.py'],check=True)
    subprocess.run([sys.executable,'code/run_extended_comparison.py','--scheme','fivefold'],check=True)
    subprocess.run([sys.executable,'code/summarize_extended.py'],check=True)
else:
    print('Using completed outputs: 3,870 actual outer fits; inner tuning fits additional.')

Using completed outputs: 3,870 actual outer fits; inner tuning fits additional.


![main_model_comparison](results/expanded/figures/main_model_comparison.png)

![paired_mae_changes](results/expanded/figures/paired_mae_changes.png)

![prediction_scatter](results/expanded/figures/prediction_scatter.png)

![high_score_bias](results/expanded/figures/high_score_bias.png)

![selection_frequency](results/expanded/figures/selection_frequency.png)

## Scope

No model, failed-looking prediction or test split was deleted. The initial experiment, which motivated this expansion, is retained under results/. Results remain internal to 65 patients; no clinical outcomes or external hospital cohort are tested. Encoder source revisions, public-checkpoint differences and preprocessing are documented in the report and data/extended_feature_provenance.json.